# Export OpenICU concepts to YAIB/RICU dynamic tables

This notebook replaces the earlier ad-hoc YAIB notebooks with one config-driven example. Edit `config/yaib_export.yml` for paths and run parameters.

It covers:

- capped 14-day export (`01_openicu_to_yaib_dyn` style, but 14 days instead of 7)
- uncapped export over all entries (`01_openicu_to_yaib_dyn_all`)
- basic main workflow (`00_main`)
- debugging of degenerate or unequal stay windows (`check_14_unequall_stays`)

In [ ]:
from pathlib import Path
import yaml
import polars as pl

from open_icu.adapter.export.yaib.transform import build_dynamic_table, write_dynamic_table
from open_icu.adapter.export.yaib.ricu_meta import RicuConceptMeta
from open_icu.adapter.export.yaib.validation import (
    scan_dynamic,
    table_summary,
    validate_dynamic_columns,
    concept_coverage,
    ricu_range_report,
    compare_dyn_to_icustays,
    debug_concept_against_output,
)

NOTEBOOK_DIR = Path.cwd()
CONFIG_PATH = NOTEBOOK_DIR / "config" / "yaib_export.yml"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

def expand(path_value):
    return Path(path_value).expanduser().resolve()

paths = cfg["paths"]
openicu = cfg["openicu"]
export = cfg["export"]
runs = cfg["runs"]
validation_cfg = cfg.get("validation", {})

CONCEPT_ROOT = expand(paths["concept_root"])
ICUSTAYS_CSV = expand(paths["icustays_csv"])
RICU_CONCEPT_DICT = expand(paths["ricu_concept_dict"]) if paths.get("ricu_concept_dict") else None
OUTPUT_DIR = expand(paths["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET = openicu.get("dataset", "mimic-iv")
VERSION = str(openicu.get("version", "1.0.0"))

COMMON_KWARGS = dict(
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    dataset=DATASET,
    version=VERSION,
    dynamic_vars=export.get("dynamic_vars"),
    aggregation_mode=export.get("aggregation_mode", "mean"),
    include_grid=export.get("include_grid", True),
    grid_end_rounding=export.get("grid_end_rounding", "floor"),
    filter_to_icu_window=export.get("filter_to_icu_window", True),
    missing_concepts=export.get("missing_concepts", "warn"),
)

print("concept_root:", CONCEPT_ROOT)
print("icustays_csv:", ICUSTAYS_CSV)
print("ricu_concept_dict:", RICU_CONCEPT_DICT)
print("output_dir:", OUTPUT_DIR)

## Run A: 14-day YAIB/RICU-like dynamic table

This creates an hourly stay grid capped at `max_hours = 336` (= 14 × 24). The maximum hour index is inclusive, matching the converter API.

In [ ]:
fourteen_cfg = runs["fourteen_days"]
OUTPUT_14D = OUTPUT_DIR / fourteen_cfg["output_name"]

write_dynamic_table(
    output_path=OUTPUT_14D,
    max_hours=fourteen_cfg.get("max_hours", 336),
    **COMMON_KWARGS,
)

dyn_14d = scan_dynamic(OUTPUT_14D)
table_summary(dyn_14d)

In [ ]:
print(validate_dynamic_columns(dyn_14d))
concept_coverage(dyn_14d).head(20)

## Run B: all entries / uncapped dynamic table

This corresponds to the earlier `01_openicu_to_yaib_dyn_all` workflow. `max_hours = null` in YAML becomes `None` in Python.

In [ ]:
all_cfg = runs["all_entries"]
OUTPUT_ALL = OUTPUT_DIR / all_cfg["output_name"]

write_dynamic_table(
    output_path=OUTPUT_ALL,
    max_hours=all_cfg.get("max_hours"),
    **COMMON_KWARGS,
)

dyn_all = scan_dynamic(OUTPUT_ALL)
table_summary(dyn_all)

In [ ]:
print(validate_dynamic_columns(dyn_all))
concept_coverage(dyn_all).head(20)

## RICU range sanity report

This checks whether final concept values are within the min/max ranges from RICU `concept-dict.json`. Values should normally already be range-filtered before aggregation.

In [ ]:
ricu_meta = RicuConceptMeta.from_json(RICU_CONCEPT_DICT)
ricu_range_report(dyn_14d, ricu_meta).filter(
    pl.col("below_ricu_min") | pl.col("above_ricu_max")
).head(50)

## Stay-window comparison

This compares observed dynamic table stay windows with windows derived from MIMIC-IV `icustays.csv.gz`. It is the notebook-friendly version of the earlier stay-window debugging.

In [ ]:
windows_14d = compare_dyn_to_icustays(
    dyn_14d,
    ICUSTAYS_CSV,
    end_rounding=export.get("grid_end_rounding", "floor"),
)
windows_14d.select([
    pl.len().alias("n_stays"),
    pl.col("diff_start").abs().max().alias("max_abs_diff_start"),
    pl.col("diff_end").abs().max().alias("max_abs_diff_end"),
    pl.col("outtime").is_null().sum().alias("n_null_outtime"),
])

In [ ]:
windows_all = compare_dyn_to_icustays(
    dyn_all,
    ICUSTAYS_CSV,
    end_rounding=export.get("grid_end_rounding", "floor"),
)
windows_all.select([
    pl.len().alias("n_stays"),
    pl.col("diff_start").abs().max().alias("max_abs_diff_start"),
    pl.col("diff_end").abs().max().alias("max_abs_diff_end"),
    pl.col("outtime").is_null().sum().alias("n_null_outtime"),
])

## Debug potentially degenerate/unequal stays

The earlier `check_14_unequall_stays` notebook investigated stays that collapsed to only `time = 0`, especially when `outtime` was missing. This cell surfaces the same class of cases for the uncapped export.

In [ ]:
suspect_stays = (
    windows_all
    .filter(
        pl.col("outtime").is_null()
        | (
            (pl.col("dyn_start") == 0)
            & (pl.col("dyn_end") == 0)
            & (pl.col("dyn_n_timepoints") == 1)
        )
    )
    .select([
        "subject_id", "hadm_id", "stay_id", "intime", "outtime", "los_hours",
        "expected_start", "expected_end", "dyn_start", "dyn_end", "dyn_n_timepoints",
        "diff_start", "diff_end",
    ])
    .sort(["outtime", "stay_id"], nulls_last=False)
)

print("suspect stays:", suspect_stays.height)
suspect_stays.head(50)

## Debug one concept end-to-end

This traces a single dynamic variable from the OpenICU concept parquet through ICU-stay mapping, hourly aggregation, and comparison to the final wide output.

In [ ]:
debug_concept = validation_cfg.get("debug_concept", "hr")
debug = debug_concept_against_output(
    concept=debug_concept,
    output_dyn=OUTPUT_14D,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_meta=ricu_meta,
    dataset=DATASET,
    version=VERSION,
    aggregate=export.get("aggregation_mode", "mean"),
    filter_to_icu_window=export.get("filter_to_icu_window", True),
)

print("concept:", debug["concept"])
print("openicu_concept:", debug["openicu_concept"])
print("concept_file:", debug["concept_file"])
debug["comparison_summary"]

In [ ]:
debug["largest_differences"].head(20)